Run the cells top to bottom.
This notebook extracts chemistry-related text from patent HTML and normalizes it into structured reaction records.

In [ ]:
pip install lxml
pip install beautifulsoup4
pip install unicodeit
pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

### Extract chemistry-related sections from the HTML page
* Install the dependencies shown in the cell above

In [3]:
import json
import os
import re
import traceback

from bs4 import BeautifulSoup
from lxml import html,etree
import unicodeit

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Keyword lists for chemistry filtering
keywords = {
    'reactant', 'product', 'equation', 'temperature', 'pressure',
    'catalyst', 'solvent', 'concentration', 'ph', 'synthesis', 'decomposition',
    'displacement', 'redox', 'polymerization', 'combustion', 'molecule',
    'bond', 'chemistry', 'reaction', 'laboratory', 'experiment', 'industrial',
    'process', 'biochemistry',"substrate","byproduct",
    "hydrogen", "isomer", "stereoisomer", "enantiomer",
    "catalyst", "activity", "selectivity",'heated','filtered','conversion','isolated','treated','afford','formation','form',
    'conversion','refluxing','purified','afforded','condition','heating','added','acidified','prepared','%','adding','compound',
    'product','heat'
}
unit_keywords = {'g','mg','l','ml','mmol','mole','pmol','nmol','μl','μg','mhz','ppm','nm','wt','mpa'}
h_tag_set = {'h1','h2','h3','h4'}
single_list_limit = 10

total_count = 0
empty_file_stat_num = 0
total_file_stat_num = 0
extract_kv_stat_num = 0
extract_v_stat_num = 0
extract_kv_no_info_file_num = 0
extract_v_no_info_file_num = 0
extract_kv_and_v_no_info_file_num = 0
result_dict = dict()
kv_list = []

def main(item_path):

    global total_count
    global empty_file_stat_num
    global total_file_stat_num
    global extract_kv_stat_num
    global extract_v_stat_num
    global extract_kv_no_info_file_num
    global extract_v_no_info_file_num
    global extract_kv_and_v_no_info_file_num

    # Process one file
    total_count += 1
    if not os.path.isfile(item_path):
        return

    result_dict["file_path"]=item_path
    total_file_stat_num += 1
    
    v_list = []
    title = ""
    has_content_title = False
    h2_content_list=[]
    div_content_list=[]
    has_h2 = False
    example_h_tag=False
    tree = html.parse(item_path,html.html_parser)
    is_empty_content = not bool(tree.xpath('//*[@id="detailMainForm:MyTabViewId:descriptionPanel"]/div[2]/child::*'))
    if is_empty_content:
        result_dict["empty_file"]=True
        empty_file_stat_num += 1
        return
    for i in tree.xpath('//*[@class="para_text"] | //h2 | //h1 | //h3 | //h4'):
        if not i.xpath('.//table'):
            # if i.xpath('.//a'):
            #     i.xpath('.//a')
            html_object = etree.tostring(i, pretty_print=True).decode('utf-8')
            cleaned_html_object = re.sub(r'\s+(?=<sub>|<sup>)', '', html_object)
            soup = BeautifulSoup(cleaned_html_object, 'html.parser')
            # Remove anchor tags
            for a_tag in soup.find_all('a'):
                a_tag.decompose()
            # Normalize superscripts and subscripts
            for sup_tag in soup.find_all('sup'):
                if sup_tag.string is None:
                    sup_tag.string = ""
                else:
                    sup_tag.string = re.sub(r'\S+', lambda match: f"^{{{match.group(0)}}}", sup_tag.string)
                    sup_tag.string=unicodeit.replace(sup_tag.string)
            for sub_tag in soup.find_all('sub'):
                if sub_tag.string is None:
                    sub_tag.string = ""
                else:
                    sub_tag.string = re.sub(r'\S+', lambda match: f"_{{{match.group(0)}}}", sub_tag.string)
                    sub_tag.string=unicodeit.replace(sub_tag.string)
            # Normalize whitespace
            row_content = re.sub(r'\s+', " ", soup.get_text()).replace("° C","°C")
            if row_content.strip() == "":
                continue
            # Keyword relevance check
            is_relevant = filter_text_by_keywords(row_content)
            # Heuristic check for reaction-like text
            num_len = row_content.count('-')+count_numbers_in_sentence(row_content)
            braces_len = row_content.count('(')+row_content.count(')')+row_content.count('=')+row_content.count(',')
            is_reaction = num_len>30 and num_len+braces_len>40 and ((num_len+braces_len)/len(row_content))>0.13
            verbs = row_content.strip().split(" ")
            if contains_only_parentheses_spaces(row_content):
                continue
            if i.tag in h_tag_set:
                # Flush the previous section before starting a new title
                collect_info(v_list,kv_list,title,h2_content_list,file_name)
                collect_info(v_list,kv_list,title,div_content_list,file_name)
                has_content_title = False
                if ('example' in row_content.lower() or 'compound' in row_content.lower()) and not row_content.count('-'):
                    example_h_tag = True
                else:
                    example_h_tag = False
                # Treat the heading as a reaction title when it looks chemical
                if ((row_content.count('-') and
                    not (row_content.count('-')==1 and
                        any(re.match(r'^\d+-\d+$',k) for k in row_content.split(' ')))) or
                    row_content.lower().count('production') or
                    row_content.lower().count('preparation') or
                    row_content.lower().count('synthesis')):
                    # Prefer hyphenated reaction titles over generic example headings
                    example_h_tag = False
                    has_h2 = True
                    title = row_content
                else:
                    has_h2 = False
            if i.tag not in h_tag_set:
                if find_valid_joint(row_content) and ((len(verbs) <=5 and row_content.count('-') and 'FAB-MS' not in row_content) or
                    (len(verbs) <=6 and row_content.count('-') and 'of' in verbs) or
                    (len(verbs) <=8 and row_content.count('-') and row_content.lower().count('synthesis')) or
                    (len(verbs) <=8 and row_content.count('-') and row_content.lower().count('preparation')) or
                    (len(verbs) <=9 and row_content.count('-') and row_content.lower().count('synthesis') and row_content.lower().count('compound')) or
                    (len(verbs) <=20 and (row_content.lower().count('production') or
                                          row_content.lower().count('preparation') or
                                          row_content.lower().count('synthesis'))
                     and ((row_content.strip().startswith('<') and row_content.strip().endswith('>')) or (row_content.strip().startswith('(') and row_content.strip().endswith(')'))))
                ):
                    collect_info(v_list,kv_list,title,h2_content_list,file_name)
                    collect_info(v_list,kv_list,title,div_content_list,file_name)
                    has_content_title = True
                    has_h2 = False
                    example_h_tag = False
                    title = row_content
                else:
                    if is_relevant or is_reaction:
                        if has_h2:
                            h2_content_list.append((True,row_content))
                        # Append reaction-like title content to the current section
                        elif has_content_title:
                            div_content_list.append((True,row_content))
                            
                        elif example_h_tag:
                            v_list.append(row_content)
                        else:
                            if is_relevant and is_reaction:
                                v_list.append(row_content)
                            pass

                    else:
                        if has_h2:
                            h2_content_list.append((False,row_content))
                        elif has_content_title:
                            div_content_list.append((False,row_content))
                      
                        pass
    if has_h2:
        collect_info(v_list,kv_list,title,h2_content_list,file_name)
    if has_content_title:
        collect_info(v_list,kv_list,title,div_content_list,file_name)
    print("======has_title_info jsonl:======\n")
    
    if len(kv_list)==0:
        result_dict["no_kv_info_file"]=True
        extract_kv_no_info_file_num += 1
    for i,kv in enumerate(kv_list):
        if kv is not None:
            extract_kv_stat_num += 1
            kv["index"]=str(i)                
            print(json.dumps(kv, ensure_ascii=False))
    print("======no_title_info jsonl:======\n")
    
    if len(v_list)==0:
        result_dict["no_v_info_file"]=True
        extract_v_no_info_file_num += 1
    for i,v in enumerate(v_list):
        extract_v_stat_num += 1
        v_dict = {"index":str(i),"file_name":file_name,"v":v}
        print(json.dumps(v_dict, ensure_ascii=False))
    if len(v_list)==0 and len(kv_list)==0:
        result_dict["no_any_info_file"]=True
        extract_kv_and_v_no_info_file_num += 1

def find_valid_joint(text):
    return bool(re.search(r'([^\d\s]+-[^\d\s]+)|([^\d\s]+-[\d]+)|([\d]+-[^\d\s]+)', text))

def collect_info(v_list,kv_list,title,content_list,file_name):
    if len(content_list) >= single_list_limit:
        v_list.extend([text for is_match,text in content_list if is_match])
    else:
        content = get_h2_content(title,content_list,file_name)
        if content is not None:
            kv_list.append(content)
    content_list.clear()

def get_h2_content(title,h2_content_list,file_name):
    if len(h2_content_list) != 0:
        if any(match for match,_ in h2_content_list):
            
            return {"file_name":file_name,"k":title,"v":' '.join([text for _,text in h2_content_list])}
    return None

def contains_only_parentheses_spaces(line):
    pattern = r'^[() ]*$'
    return re.match(pattern, line) is not None

def filter_text_by_keywords(text):
    lemmatizer = WordNetLemmatizer()
    lower_text = text.lower()
    # Tokenize the text
    words = word_tokenize(lower_text)
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in words]
    one_filter = set(lemmatized_tokens).intersection(keywords)
    two_filter = set(lemmatized_tokens).intersection(unit_keywords)
    yield_count = 1 if "yield" in set(lemmatized_tokens) and "%" in lower_text else 0
    unit_count = len(two_filter)+lower_text.count("°")+yield_count
    if len(one_filter)>=2 and (unit_count!=0):
        return True
    else:
        if 'nmr' in lemmatized_tokens and unit_count!=0:
            return True
        else:
            return False

def count_numbers_in_sentence(sentence)->int:
    # Match integers and decimals
    pattern = r'\d+(\.\d+)?'
    matches = re.findall(pattern, sentence)
    # Return the number of numeric tokens
    return len(matches)

if __name__ == '__main__':
    
    input_file="./input/test.html"
    file_name = os.path.splitext(os.path.basename(input_file))[0]
    try:
        
        main(input_file)
        
    except Exception as e:
        print(f"the file is {input_file},the error of main method is:",e)
        traceback.print_exc()

    result_dict["extract_kv_num"] = extract_kv_stat_num
    result_dict["extract_v_num"] = extract_v_stat_num
    print("======stat_info json:======\n")
    print(json.dumps(result_dict,ensure_ascii=False))


======has_title_info jsonl:======

{"file_name": "test", "k": "Synthesis of Examples 1-4 (silyl-fluorene fluorophores) ", "v": " The synthesis for Example 1 was adapted from the method disclosed in Wei, W.; Djurovich, P. I.; Thompson, M. E. Chem. Mater. 2010, 22, 1724-1731, which is incorporated herein by reference. 2-bromo-9,9-dimethylfluorene was used as received from Oakwood Chemicals, (which can also be prepared according to a known procedure (J. Phys. Chem. Lett. 2010, 1, 616-620; incorporated herein by reference)).   A dry 500 mL round bottom flask was charged with a stir bar, 2-bromo-9,9-dimethylfluorene (14.57 g, 53.33 mmol, 2.0 equiv) and THF (89 mL) under argon. The mixture was cooled to −78°C., followed by addition of n-BuLi as 2.5 M solution in hexanes (21.33 mL, 2.0 equiv) via syringe over 10 minutes. A dark red slurry formed. The mixture was stirred for 30 minutes, followed by the addition of Ph₂SiCl₂ (6.75 g mL, 26.66 mmol) dropwise via syringe. The mixture was slowly wa

### Parse reactants and products into structured JSON
* Requires Ray; use Python 3.12 in the `syn-rrag-train` environment
* GPU inference is required

In [ ]:
pip install "ray[default]"
pip install "ray[llm]" 

In [ ]:
import ast
import json
import os
import re
import time
import csv

from enum import Enum

import ray
from ray.data import Dataset,DataContext
from typing_extensions import override
from ray.data.llm import build_llm_processor
from ray.llm._internal.batch.processor import vLLMEngineProcessorConfig
import sys

model="mistralai/Devstral-Small-2505"


# Processing status enum
class Status(Enum):
    INIT = 1
    SUCCESS = 2
    UNMATCH = 3
    REQUEST_ERR = 4
    PARSE_ERR = 5
    CLEAN_ERR = 6


config = vLLMEngineProcessorConfig(
    model="mistralai/Devstral-Small-2505",
    engine_kwargs={
        "enable_chunked_prefill": True,
        "max_num_batched_tokens": 48000,
        "max_model_len": 112784,
        "tokenizer_mode": "mistral",
        "config_format": "mistral",
        "load_format": "mistral",
    },
    runtime_env={"env_vars": {"HF_ENDPOINT": "https://hf-mirror.com","HF_HUB_ENABLE_HF_TRANSFER":"0","device":"cuda"}},
    concurrency=1,
    batch_size=128,
    tokenize = False,
    detokenize = False,
    apply_chat_template=False
)
def preprocess(row):
    prompt_str = '<s>[SYSTEM_PROMPT]You are a chemistry expert.The user will provide you with the content list of a chemical reaction, you are asked to analyse each content of the reaction in list and extract the key information ,if the information of value is not provided in input content,give a empty string,and output it as JSON list with no additional information, or empty json if the input content is not about chemical reaction,Please check that each json in the output json list is fully compliant with the following formatting integrity, and make sure that the entire json list is wrapped in the markdown format for json style,and ensure json can be parsed properly,each output JSON should follow the format below:[{"reactants": [{"name":"<name of reactant>","volume":"<volume of reactant>"}],"products": [{"name":"<name of product>","volume":"<volume of product>"}],"reagents": [{"name":"<name of reagent>","volume":"<volume of reagent>"}],"solvents": [{"name":"<name of solvent>","volume":"<volume of solvent>"}],"additives": [{"name":"<name of additive>","volume":"<volume of additive>"}],"reaction_type": ["<reaction_type>"], "yield": "<yield>", "is_multi_stage": <Whether the reaction is divided into multiple stages>,"nmr":"<the information of NMR,it is a string not a object>"},<each one in input list should output a json object>][/SYSTEM_PROMPT][INST]'+f'{row["question"]}'+'[/INST]'
    return dict(
        sampling_params=dict(
            n=1,  # Generate one completion
            temperature=0.5,  # Lower temperature for more deterministic outputs
            skip_special_tokens=True,  # Skip special tokens
            max_tokens=5024,
            min_tokens=35
        ),
        prompt=prompt_str,
        max_new_tokens=2048,
    )


# Post-processing
def postprocess(row:dict)->dict:
    def remove_comments(json_str):
        # Remove // comments
        no_comments_str = re.sub(r'//[^"]*?[^,"]\n', '', json_str)
        return no_comments_str

    def parse_json(json_str):
        # Extract and parse JSON data
        try:
            match_value = json_str.replace("\\","").replace('[OUT]',"").replace('[/OUT]',"").replace('[/INST]',"").replace('[INST]',"")
            json_data_arr = json.loads(match_value)
        except Exception as ex_1:
            row["status"]=Status.PARSE_ERR.name
            row["err"] = str(ex_1)
        else:
            if isinstance(json_data_arr,list):
                row["response"]=json.dumps(json_data_arr, ensure_ascii=False)
                row["status"]=Status.SUCCESS.name
            elif isinstance(json_data_arr,dict):
                row["response"]=json.dumps([json_data_arr], ensure_ascii=False)
                row["status"]=Status.SUCCESS.name

            else:
                row["status"]=Status.PARSE_ERR.name
                row["err"]="unknown type,response only support arr and dict now."
    try:
        row["response"]=row["generated_text"]
    except Exception as ex:
        row["status"]=Status.REQUEST_ERR.name
        row["err"] = str(ex)
        return row
    # Match the JSON block with a regular expression
    json_pattern = re.compile(r'```json(.*?)```', re.DOTALL)
    match = json_pattern.search(row["response"])
    if match:
        parse_json(remove_comments(match.group(1)))
    else:
        json_content = row["response"].strip().strip("```")
        try:
            no_comments_json_content = remove_comments(json_content)
            json.loads(no_comments_json_content.replace("\\","").replace('[OUT]',"").replace('[/OUT]',"").replace('[/INST]',"").replace('[INST]',""))
        except Exception:
            row["status"]=Status.UNMATCH.name
            row["err"]="can not match json_pattern"
        else:
            parse_json(no_comments_json_content)
    return row



# Clean compound names
def clean_name(row:dict)->dict:
    pattern = r'\s+(\([^)]+\)|\[[^]]+\]|\d+[a-z]*|\d+\s*[a-z-A-Z]*)\s*$'
    if row["status"] != Status.SUCCESS.name:
        return row
    try:
        response = json.loads(row["response"])
        for clear_json in response:
            clear_json["reactants"]=[{"name":re.sub(pattern, '', c["name"].strip()).strip(),"volume":c["volume"] if "volume" in c else ""} for c in clear_json["reactants"]] if "reactants" in clear_json else []
            clear_json["products"]=[{"name":re.sub(pattern, '', c["name"].strip()).strip(),"volume":c["volume"] if "volume" in c else ""} for c in clear_json["products"]] if "products" in clear_json else []
            clear_json["reagents"]=[{"name":re.sub(pattern, '', c["name"].strip()).strip(),"volume":c["volume"] if "volume" in c else ""} for c in clear_json["reagents"]] if "reagents" in clear_json else []
            clear_json["solvents"]=[{"name":re.sub(pattern, '', c["name"].strip()).strip(),"volume":c["volume"] if "volume" in c else ""} for c in clear_json["solvents"]] if "solvents" in clear_json else []
            clear_json["additives"]=[{"name":re.sub(pattern, '', c["name"].strip()).strip(),"volume":c["volume"] if "volume" in c else ""} for c in clear_json["additives"]] if "additives" in clear_json else []
        row["clean_response"] = json.dumps(response, ensure_ascii=False)
    except Exception as ex:
        row["err"] = str(ex)
        row["status"] = Status.CLEAN_ERR.name
    return row

# Input base class
class Input:
    def __init__(self,kv_list):
        self.kv_list = kv_list
    def read(self):
        return ray.data.from_items(self.kv_list)
    @staticmethod
    def get_valid_row(row:dict) -> bool:
        return True
    @staticmethod
    def prepare_data(row:dict)->dict:
        return row
    def get_data(self)->Dataset:
        return self.read().filter(self.get_valid_row).map(self.prepare_data)

# First-pass input adapter
class FirstProcess(Input):
    @override
    def read(self):
        def process_line(line):
            # Serialize one input record
            return {"text": line}
        def generate_processed_data(input_list):
            return [process_line(json.dumps(line, ensure_ascii=False)) for line in input_list]
        return ray.data.from_items(generate_processed_data(self.kv_list))

    @staticmethod
    def get_valid_row(row:dict) -> bool:
        return True

    @staticmethod
    def prepare_data(row:dict)->dict:
        line_obj = ast.literal_eval(row["text"].strip())
        question = f'{line_obj["k"].strip()}.  this is the product: {line_obj["v"].strip()}'

        line_obj["file_name"] = "" if "file_name" not in line_obj else line_obj["file_name"]
        row["file_name"]=line_obj["file_name"]
        row["index"]=line_obj["index"]
        row["question"]=question
        row["response"]=""
        row["clean_response"]=""
        row["status"]=Status.INIT.name
        row["err"]=""
        return row


def run():
    """
    Run the LLM extraction pipeline on the prepared records.
    :return: Processed row list
    """

    filtered_dataset = FirstProcess(kv_list).get_data()
    processor = build_llm_processor(
        config,
        preprocess=preprocess,
        postprocess=postprocess,
    )
    result_dataset = processor(filtered_dataset)
    row_list = result_dataset.map(clean_name).select_columns(["text","file_name","index","question","response","clean_response","status","err"]).take_all()
    for row in row_list:
        print(row)
    return row_list



if __name__ == '__main__':
    try:
        start_time = time.time()
        row_list = run()
        total_count = len(row_list)
        end_time = time.time()
        print(f"Processed {total_count} reactions in {end_time-start_time:.2f}s ({((end_time-start_time)/total_count):.2f}s per reaction)")
    except Exception as e:
        print("Async task failed:",e)
        traceback.print_exc()

In [ ]:
### Reshape the output schema
* Expected record format:
```
{
    "reactants": Array,
    "products": Array,
    "reagents": Array,
    "solvents": Array,
    "additives": Array,
    "reaction_type": Array,
    "yield": String,
    "is_multi_stage": Boolean,
    "nmr": String,
    "index": String,
    "file_name": String
}
```

In [ ]:
import json

def extract_and_merge_clean_response(input_data):
    output_data = []
    for row in input_data:
        file_name = row.get('file_name', '')
        index = row.get('index', '')
        clean_response_str = row.get('clean_response', '')
        if not clean_response_str:
            continue
        try:
            # Parse the JSON list
            clean_response_list = json.loads(clean_response_str)
            if isinstance(clean_response_list, list):
                for item in clean_response_list:
                    item["index"] = index
                    item["file_name"] = file_name
                    if isinstance(item["reaction_type"], str):
                        item["reaction_type"] = [item["reaction_type"]]
                    # Emit one JSON object per reaction
                    output_data.append(json.dumps(item, ensure_ascii=False))
            else:
                clean_response_list["index"] = index
                clean_response_list["file_name"] = file_name
                if isinstance(clean_response_list["reaction_type"], str):
                    clean_response_list["reaction_type"] = [clean_response_list["reaction_type"]]
                # Emit a single JSON object when the payload is not a list
                output_data.append(json.dumps(clean_response_list, ensure_ascii=False))
            print("Structured reaction output:")
            print(output_data)
        except Exception as e:
            # Skip rows that fail to parse
            print(e)
            continue

if __name__ == "__main__":
    extract_data = extract_and_merge_clean_response(row_list)

### Convert compound names to SMILES
* PyOpsin requires Java; install Java 21 and set `JAVA_HOME`

In [ ]:
pip install pyopsin

In [ ]:
import json
from typing import List, Dict
import sys
from pyopsin.pyopsin import PyOpsin

class Reaction:
    def __init__(self, index: str = None, file_name: str = None,
                 reactants: List[Dict[str, str]] = None,
                 products: List[Dict[str, str]] = None,
                 reagents: List[Dict[str, str]] = None,
                 solvents: List[Dict[str, str]] = None,
                 additives: List[Dict[str, str]] = None,
                 reaction_type: List[str] = None,
                 yield_: str = None,
                 is_multi_stage: bool = None,
                 nmr: str = None):
        self.index = index
        self.file_name = file_name
        self.reactants = reactants
        self.products = products
        self.reagents = reagents
        self.solvents = solvents
        self.additives = additives
        self.reaction_type = reaction_type
        self.yield_ = yield_
        self.is_multi_stage = is_multi_stage
        self.nmr = nmr

    @staticmethod
    def from_dict(d: dict) -> 'Reaction':
        return Reaction(
            index=d.get('index'),
            file_name=d.get('file_name'),
            reactants=d.get('reactants'),
            products=d.get('products'),
            reagents=d.get('reagents'),
            solvents=d.get('solvents'),
            additives=d.get('additives'),
            reaction_type=d.get('reaction_type'),
            yield_=d.get('yield'),
            is_multi_stage=d.get('is_multi_stage'),
            nmr=d.get('nmr')
        )

    def to_dict(self) -> dict:
        return {
            'index': self.index,
            'file_name': self.file_name,
            'reactants': self.reactants,
            'products': self.products,
            'reagents': self.reagents,
            'solvents': self.solvents,
            'additives': self.additives,
            'reaction_type': self.reaction_type,
            'yield': self.yield_,
            'is_multi_stage': self.is_multi_stage,
            'nmr': self.nmr
        }

def parse_smiles(chemical_map: dict) -> dict:

    chemical_name = chemical_map.get("name")

    try:
        smiles = PyOpsin().to_smiles(chemical_name)
    except Exception as e:
        print(f"Error parsing chemical name '{chemical_name}': {str(e)}")
        smiles = None

    if smiles is None:
        print(f"Warning: could not convert '{chemical_name}' locally; verify the name or use a lookup service")
        smiles = ""

    chemical_map["name"] = smiles
    return chemical_map

def trans_json(chemical_json: dict) -> str:
    if not chemical_json:
        return ""

    try:
        reaction = Reaction.from_dict(chemical_json)

        # Convert all compound lists
        for attr in ['reactants', 'products', 'reagents', 'solvents', 'additives']:
            if getattr(reaction, attr):
                setattr(reaction, attr,
                        [parse_smiles(item) for item in getattr(reaction, attr)])

        return reaction.to_dict()
    except Exception as e:
        print(f"Error processing JSON: {str(e)}")
        return ""


if __name__ == "__main__":
    print("\nConverted SMILES records:")
    smiles_list = []
    for i in extract_data:
        smile_result = trans_json(i)
        print(smile_result)
        smiles_list.append(smile_result)

### Validate and canonicalize SMILES

* 1. Canonicalize SMILES strings

* 2. Derive `yield_type` from the `yield` field:  
    Original field (`yield`) -> new field (`yield_type`)  
    None, ""        --> unknown;  
    a%,0.xx         --> percent;  
    mg,g, ...       --> mass
  
* 3. Add `smiles_valid` using the compound fields:  
    `smiles_valid` is `False` if any of these lists contains an empty `name`:  
    (`reactants`, `products`, `reagents`, `solvents`, `additives`);  
    `smiles_valid` is also `False` if either of these lists is missing:  
    (`reactants`, `products`);  
    Otherwise, set `smiles_valid` to `True`


In [ ]:
from rdkit import Chem
from rdkit.Chem import MolToSmiles
import json
import os
import re

def get_yield_type(yield_tag:str)->str:
    if yield_tag is None or yield_tag == "":
        return "unknown"
    pattern_one=r"^0(.\d+){0,1}$"
    pattern_two=r"^1(.0+){0,1}$"
    if re.match(pattern_one, yield_tag) or re.match(pattern_two, yield_tag) or "%" in yield_tag:
        return "percent"
    else:
        return "mass"

def contain_empty(json_obj,col_name):
    for info in json_obj[col_name]:
        if info["name"] == "":
            return True
    return False

def normalize_smiles(smiles:str)->str:
    if smiles is None or smiles == "":
        return ""
    try:
        # Build an RDKit molecule from SMILES
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        # Return canonical SMILES
        normalized_smiles = MolToSmiles(mol, canonical=True, isomericSmiles=True)
        return normalized_smiles
    except Exception:
        return ""

def tranfer_record(smiles_obj):
    try:        
        smiles_obj["yield_type"] = get_yield_type(smiles_obj["yield"])
        cols = ["reactants","products","reagents","solvents","additives"]
        for col in cols:
            if col in smiles_obj:
                if isinstance(smiles_obj[col],list):
                    for obj in smiles_obj[col]:
                        if "name" in obj:
                            obj["name"] = normalize_smiles(obj["name"])
        if (
            contain_empty(smiles_obj,'reactants') or
            contain_empty(smiles_obj,'products') or
            contain_empty(smiles_obj,'reagents') or
            contain_empty(smiles_obj,'solvents') or
            contain_empty(smiles_obj,'additives')
        ):
            smiles_obj["smiles_valid"]=False
        elif len(smiles_obj['reactants'])==0 or len(smiles_obj['products'])==0:
            smiles_obj["smiles_valid"]=False
        else:
            smiles_obj["smiles_valid"]=True
        return smiles_obj
    except Exception as e:
        print(e)
        return None

if __name__ == '__main__':
    print("="*50)
    smile_valid_result_arr=[]
    for p4_single_smiles in smiles_list:
        tranfer_record_json = tranfer_record(p4_single_smiles)
        if tranfer_record_json is None:
            continue
        smile_valid_result_arr.append(tranfer_record_json)
    
    print("Records after SMILES canonicalization and yield/smiles validation:")
    print(smile_valid_result_arr)